# K513 · Week 5, Session 1
## Classification — logistic regression

Last week you built a model that predicted a number: what is this house worth. This week the
answer is a category — yes or no — and the question behind it is a decision.

You run maintenance for a plant floor. There are 689 machines on record, five sensor readings for
each of them, and a record of which ones failed. Your crew can pull **twenty machines** off the
line tonight. Today is about choosing which twenty.

---

### Before you type anything

**File → Save a copy in Drive.**

This notebook is read-only for you. You can type into it and run it and it will look completely
normal, but nothing you do will be saved. Save your own copy first, every time.

---

### Using AI in this notebook

Gemini is built into Colab and you are welcome to use it here. Two things worth knowing:

- It does not know which columns you have or what we covered in class. Whatever it writes, you own.
- The most useful thing you can ask it is **"explain what this line does"** — not "write it for me".

Today's trap: ask an AI to "predict which machines will fail" and it will hand you `predict()`,
which returns a yes or a no for every machine. It has no way of knowing that your crew can only
do twenty tonight, so it cannot know that what you actually need is a **ranking**. That judgment
is yours, and it is the whole session.

---

### Turn off Unwanted AI Assistance

AI-powered coding completion is turned on by default. It is convenient but does not give you a chance
to think and learn. Turning it off helps you learn. You can always turn it back on when needed.
- Tools → settings → AI Assistance → Uncheck "Show AI-powered inline code completions"
- Tools → settings → Uncheck "Show context-powered code completions"

---

### How to run a cell

Click on a cell, then press **Shift + Enter**. That runs it and moves you to the next one.

## Setup

Everything below is the same stack you have been using since Week 1, plus one new import:
`LogisticRegression`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pd.set_option('display.precision', 3)
np.set_printoptions(precision=3, suppress=True)

RANDOM_SEED = 42
MACHINE_URL = 'https://raw.githubusercontent.com/jl-uscn/k513-data/main/Machine%20Failure%20Data.csv'

In [ ]:
machine_df = pd.read_csv(MACHINE_URL)
print(machine_df.shape)
machine_df.head()

### What is in the table

| Column | What it is |
|---|---|
| `Product ID` | a unique code for each machine |
| `Type` | the machine class: `L`, `M` or `H` |
| `Air Temp` | ambient air temperature around the machine, in kelvin |
| `Proc Temp` | processor temperature inside the machine, in kelvin |
| `Rtn Speed` | rotation speed, in rpm |
| `Torque` | rotational force, in newton-meters |
| `Tool Wear` | accumulated wear on the tool, in minutes |
| `Failure?` | whether the machine failed — `Yes` or `No` |

In [ ]:
machine_df.info()

In [ ]:
machine_df.describe()

### The target variable

Two things to look at before anything else: what the target column actually contains, and how the
two classes are split.

In [ ]:
machine_df['Failure?'].value_counts(normalize=True)

**Notice the split.** About half these machines failed. A real maintenance log does not look like
that — failure is rare. This table is a *balanced sample*: every failure on record, plus a matching
number of healthy machines. It makes the arithmetic today clean and honest. Hold on to the fact that
it is not what you would pull out of a plant database.

In [ ]:
sns.pairplot(machine_df, hue='Failure?', palette=['#2E5EA8', '#990000'],
             vars=['Air Temp', 'Proc Temp', 'Rtn Speed', 'Torque', 'Tool Wear'],
             plot_kws={'s': 14, 'alpha': 0.5})
plt.show()

In [ ]:
sns.countplot(data=machine_df, x='Type', hue='Failure?',
              palette=['#2E5EA8', '#990000'])
plt.show()

**Read the two pictures before you model anything.** In the scatterplot matrix, look down the
`Torque` and `Rtn Speed` rows — the red and blue clouds separate. In the count plot, they do not
separate much at all. That is a prediction you can check in a few minutes.

---
# 1 · Build the classifier

## Setting up X and y

Two changes from last week, and only two.

In [ ]:
# 1 is the class you are looking for. Here that is a machine that failed.
y = (machine_df['Failure?'] == 'Yes').astype(int)

X = machine_df.drop(['Failure?', 'Product ID'], axis=1)

continuous_features = ['Air Temp', 'Proc Temp', 'Rtn Speed', 'Torque', 'Tool Wear']
categorical_features = ['Type']

print('rows:', len(y), '  failures:', y.sum(), '  healthy:', (y == 0).sum())

**Why write the comparison out instead of using a tool for it?** Because *which class counts as 1*
is a business decision. It is the thing you are trying to find, and it is almost never the common
one. Writing it out means you have chosen it on purpose.

## Partitioning the data

Same call as last week, with one argument added.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y)

print('training rows:', len(X_train), '  test rows:', len(X_test))
print('share that failed - training:', round(y_train.mean(), 3),
      '  test:', round(y_test.mean(), 3))

`stratify=y` makes the training set and the test set carry the same mix of failures as the original
table. On a table that is already half and half it changes almost nothing. On a table where the
class you care about is 3% of the rows, a random split can hand the test set almost none of them —
and then the score you report is about a handful of machines. Set it on every classification split.

## Preprocessing, and the model

This block is last week's, unchanged: scale the numeric columns, one-hot encode the text column,
chain the whole thing to an estimator with a `Pipeline`.

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), continuous_features),
    ('cat', OneHotEncoder(drop='first'), categorical_features)])
preprocessor

---
### ✏️ Now You Try · 1

**(a)** One word is blanked out below. Fill it in and fit the model.

In [ ]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', ____(random_state=RANDOM_SEED))])

model.fit(X_train, y_train)

**(b)** Print the training accuracy and the test accuracy. `score()` still works; on a classifier it
returns accuracy — the share of machines the model called correctly.

In [ ]:
print('training accuracy:', round(model.score(X_train, y_train), 3))
print('    test accuracy:', round(model.____(X_test, y_test), 3))

**(c)** Work out the baseline. If you knew nothing about a machine except how the training column of
Yes and No is split, you would call every machine the class that comes up most often. What share of
the *test* machines would that get right?

Careful with which rows you are allowed to look at.

In [ ]:
majority_class = y_train.____()[0]
baseline_accuracy = (y_test == majority_class).mean()

print('majority class in training data:', majority_class)
print('baseline accuracy on test:', round(baseline_accuracy, 3))

**(d)** Run the **evaluation steps** on your two scores. Which step are you at, and what does it
tell you to do next?

*Your answer:*

**(e)** In your own words: what does the test accuracy mean, counted in machines? There are 173
machines in the test set.

*Your answer:*

---
# 2 · The probability is the product

A classifier has two ways of answering. They are not interchangeable.

In [ ]:
# predict() gives you one answer per machine
print('predict():      ', model.predict(X_test)[:6])

# predict_proba() gives you two columns. The second is the chance of 1 - the chance it fails.
print('predict_proba():')
print(model.predict_proba(X_test)[:6])

The two columns add up to 1, so the second one carries everything. And `predict()` is just that
second column with a cut applied at 0.5 — a default that came with the library, not a decision
anybody made about your plant.

---
### ✏️ Now You Try · 2

**(a)** Pull out the failure probability for every test machine — the second column of
`predict_proba`.

In [ ]:
risk = model.predict_proba(X_test)[:, ____]
print(risk[:6].round(3))

**(b)** Put the risk score next to what actually happened, and sort the table with the highest risk
at the top. Keeping the two in one DataFrame is what stops them getting out of step.

In [ ]:
shift = pd.DataFrame({
    'risk': risk,
    'failed': y_test.values,
}, index=X_test.index)

shift = shift.sort_values('risk', ascending=____)
shift.head(10)

**(c)** Of the twenty machines at the top of that list, how many really failed? And of the twenty at
the bottom?

In [ ]:
print('top 20    - really failed:', shift.head(20)['failed'].sum(), 'of 20')
print('bottom 20 - really failed:', shift.____(20)['failed'].sum(), 'of 20')
print()
print('for comparison, 20 machines picked at random would average:',
      round(20 * y_test.mean(), 1))

**(d)** Your crew can do twenty machines tonight. In your own words: what do you hand them, and what
do you tell the plant manager that list is worth? Name a quantity — "it is better than random" is
not an answer.

*Your answer:*

---
# 3 · Where you cut, and what the model is made of

`predict()` cut the list at 0.5. Nothing says you have to. The two helpers below are written for
you — you are not expected to be able to write them, and neither is on any assessment. What matters
is what they tell you.

In [ ]:
def at_cut(cut, table=None):
    """What happens to the shift if you pull every machine at or above `cut`."""
    table = shift if table is None else table
    pulled = table[table['risk'] >= cut]
    total_failures = int(table['failed'].sum())
    return pd.Series({
        'cut': cut,
        'machines pulled': len(pulled),
        'of those, really failed': int(pulled['failed'].sum()),
        'share really failed': round(pulled['failed'].mean(), 3) if len(pulled) else np.nan,
        'failures caught': f"{int(pulled['failed'].sum())} of {total_failures}",
        'accuracy': round(((table['risk'] >= cut).astype(int) == table['failed']).mean(), 3),
    })


pd.DataFrame([at_cut(c) for c in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]]).set_index('cut')

**Read the last two columns against each other.** Moving the cut up buys you a cleaner list and
costs you failures you never looked at. Moving it down does the reverse. Then look at `accuracy`:
it barely moves across the whole range. The single number the model reports is nearly blind to the
decision you are actually making.

In [ ]:
# Which way does each column push, and how hard?
feature_names = (continuous_features
                 + list(model.named_steps['preprocessor']
                        .named_transformers_['cat']
                        .get_feature_names_out(categorical_features)))

coefficients = pd.Series(model.named_steps['classifier'].coef_[0],
                         index=feature_names).sort_values()

colors = ['#990000' if c > 0 else '#2E5EA8' for c in coefficients]
ax = coefficients.plot(kind='barh', color=colors)
ax.bar_label(ax.containers[0], fmt='%+.2f', padding=4)
ax.set_xlim(-1.9, 3.9)
ax.set_xlabel('pushes toward failure  →')
plt.title('Which way does each column push?')
plt.show()

Every column was put on the same scale before fitting, so these bars can be compared with each
other. Read the **sign** and the **order** — not the size as a percentage of anything.

For a magnitude you can put in a memo, ask the model directly.

In [ ]:
def what_if(machine, column, values):
    """Change one reading on one machine and ask the model again."""
    rows = []
    for v in values:
        altered = machine.copy()
        altered[column] = v
        rows.append({column: v,
                     'chance of failing': round(model.predict_proba(altered)[0, 1], 3)})
    return pd.DataFrame(rows)


machine_47 = pd.DataFrame([{
    'Type': 'L', 'Air Temp': 300.0, 'Proc Temp': 310.0,
    'Rtn Speed': 1450, 'Torque': 40.0, 'Tool Wear': 100}])

what_if(machine_47, 'Torque', [30, 40, 50, 60])

Between 40 and 50 the answer goes from unlikely to a coin flip. There is no single number that
describes what torque is worth, which is exactly what the bend in the curve bought us.

**Every reading in `machine_47` sits inside the range the model was trained on.** Ask it about a
machine running at an air temperature no machine in the data ever reached and it will still return a
number, and that number will be worth nothing.

---
### ✏️ Now You Try · 3

**(a)** Do the same thing with tool wear. Use `machine_47` and try 0, 100 and 200 minutes.

In [ ]:
what_if(____, 'Tool Wear', [0, 100, 200])

**(b)** Tomorrow the crew can only do **ten** machines, not twenty. Choose a cut point that gets the
list to about ten, and check it with `at_cut`. Try a few values.

In [ ]:
pd.DataFrame([at_cut(c) for c in [____, ____, ____]]).set_index('cut')

**(c)** Write about **120 words** to the plant manager. Three things have to be in it:

1. which machines you want pulled tomorrow, and how you chose them
2. why you chose that cut point rather than a higher or lower one
3. one thing this model cannot tell them

There is no single right answer to (c). What matters is whether your reasons match the numbers
in your own table.

*Your answer:*

---
## The dial that was already on

One last thing, and it takes thirty seconds. `LogisticRegression()` arrives with a penalty already
switched on, at `C=1.0`. `C` is last Thursday's dial wired backwards: **small `C` means a strong
penalty**.

In [ ]:
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    m = Pipeline(steps=[('preprocessor', preprocessor),
                        ('classifier', LogisticRegression(C=C, random_state=RANDOM_SEED))])
    m.fit(X_train, y_train)
    print(f'C = {C:<7} train {m.score(X_train, y_train):.3f}   test {m.score(X_test, y_test):.3f}')

From 0.1 to 100 the test score does not move at all. With 516 training rows against six columns
there is nothing to regularize — that is **evaluation step 4**, *stop*. Turn it down to 0.001 and
both scores fall together, which is step 2.

A penalty is a tool for a situation, not a step in a recipe.

And the rule from Thursday still holds: if you went shopping along that column for the best test
score, the test score would stop being an honest estimate of anything.

---
## Takeaways

- The build is last week's, with **`LogisticRegression()`** in place of `LinearRegression()`,
  **`stratify=y`** added to the split, and `score()` now returning **accuracy**.
- **Which class counts as 1 is your decision.** Write it out.
- **`predict()` decides. `predict_proba()` informs.** The probability is the product, because
  sorting by it is what tells you where to spend a budget you do not have enough of.
- **The cut point is a business decision**, not a statistical one. Before you choose it, say what
  being wrong costs in each direction.
- Read coefficients for **sign and order**. For a magnitude anybody can act on, run a **what-if**.

### Steps to Act on a Model's Output

1. The model returns a probability, not an answer.
2. Sort by it. The ranking is the product.
3. The cut point is a business decision. 0.5 is a default that came with the library.
4. Before you choose it, say what being wrong costs in each direction.